# Week 4 - Noise Sensitivity Analysis

## Gaussian Noise Injection

In [4]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [5]:
df = pd.read_csv("../data/cleaned_ai4i.csv")

print(df.shape)
df.head()

(10000, 15)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_enc
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,2
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,1
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,1
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,1
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,1


In [6]:
target = "Machine failure"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
def inject_noise(X_test, noise_level):

    X_noisy = X_test.copy()

    numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns

    # Convert numeric columns to float
    X_noisy[numeric_cols] = X_noisy[numeric_cols].astype(float)

    noise = np.random.normal(
        loc=0,
        scale=noise_level,
        size=X_noisy[numeric_cols].shape
    )

    X_noisy.loc[:, numeric_cols] += noise

    return X_noisy

In [10]:
X_noise = inject_noise(X_test,0.05)

print(X_noise.head())

              UDI Product ID Type  Air temperature [K]  \
2997  2998.022234     L50177    L           300.575019   
4871  4871.990850     L52051    L           303.711495   
3858  3859.065908     L51038    L           302.519208   
951    951.915824     H30365    H           295.555625   
6463  6463.918917     H35877    H           300.545095   

      Process temperature [K]  Rotational speed [rpm]  Torque [Nm]  \
2997               309.754973             1344.940573    62.679154   
4871               312.364558             1513.012845    40.092986   
3858               311.463201             1558.980870    37.632394   
951                306.299588             1508.947236    35.839214   
6463               310.068679             1358.040991    60.450671   

      Tool wear [min]       TWF       HDF       PWF       OSF       RNF  \
2997       153.113516  0.066995  0.096371  0.017468  0.002716  0.020570   
4871       135.016111  0.023101 -0.049214 -0.163292  0.023794 -0.046298   
3858 

In [11]:
print("Original Test Data:")
print(X_test.head())

print("\nNoisy Test Data:")
print(X_noise.head())

Original Test Data:
       UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
2997  2998     L50177    L                300.5                    309.8   
4871  4872     L52051    L                303.7                    312.4   
3858  3859     L51038    L                302.5                    311.4   
951    952     H30365    H                295.6                    306.3   
6463  6464     H35877    H                300.5                    310.0   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  TWF  HDF  PWF  \
2997                    1345         62.7              153    0    0    0   
4871                    1513         40.1              135    0    0    0   
3858                    1559         37.6              209    0    0    0   
951                     1509         35.8               60    0    0    0   
6463                    1358         60.4              102    0    0    0   

      OSF  RNF  Type_enc  
2997    0    0         1  
4871  

In [13]:
numeric_cols = X_test.select_dtypes(include=[np.number]).columns

difference = X_noise[numeric_cols] - X_test[numeric_cols]

print(difference.head())

           UDI  Air temperature [K]  Process temperature [K]  \
2997  0.022234             0.075019                -0.045027   
4871 -0.009150             0.011495                -0.035442   
3858  0.065908             0.019208                 0.063201   
951  -0.084176            -0.044375                -0.000412   
6463 -0.081083             0.045095                 0.068679   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]       TWF  \
2997               -0.059427    -0.020846         0.113516  0.066995   
4871                0.012845    -0.007014         0.016111  0.023101   
3858               -0.019130     0.032394         0.017894  0.002285   
951                -0.052764     0.039214         0.024017 -0.008405   
6463                0.040991     0.050671        -0.052768  0.002693   

           HDF       PWF       OSF       RNF  Type_enc  
2997  0.096371  0.017468  0.002716  0.020570  0.007141  
4871 -0.049214 -0.163292  0.023794 -0.046298 -0.061393  
3858  0.002